In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing(as_frame=True, return_X_y=True)
df, target = data[0], data[1] 
df = StandardScaler().fit_transform(df)

cv = KFold(n_splits=4, shuffle=True, random_state=6)

In [2]:

from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd
from scipy.stats import binom

model1 = cross_val_predict(
    estimator=Ridge(alpha=1),
    X=df,
    y=target,
    cv=cv
) 


model2 = cross_val_predict(
    estimator=Ridge(alpha=10),
    X=df,
    y=target,
    cv=cv
) 

y = np.asarray(target).ravel()
pred1 = np.asarray(model1).ravel()
pred2 = np.asarray(model2).ravel()

if not (len(y) == len(pred1) == len(pred2)):
    raise ValueError("Размеры target и предсказаний не совпадают")

error1 = (pred1 - y) ** 2
error2 = (pred2 - y) ** 2

errors = pd.DataFrame(
    {"squared_error_model1": error1, "squared_error_model2": error2},
    index=target.index if hasattr(target, "index") else None,
)
display(errors)

# Нулевая разность — ничья; такие объекты не входят в критерий знаков.
diff = error1 - error2
non_ties = diff != 0
n = int(non_ties.sum())
n_model1_worse = int((diff[non_ties] > 0).sum())

print(f"Медиана ошибки model1: {np.median(error1):.6g}")
print(f"Медиана ошибки model2: {np.median(error2):.6g}")
print(f"Ненулевых разностей: {n}")
print(f"Model1 хуже: {n_model1_worse}; model2 хуже: {n - n_model1_worse}")

alpha = 0.05
if n == 0:
    print("Критерий неприменим: все парные разности равны нулю.")
else:
    # При H0 число положительных разностей имеет распределение Bin(n, 0.5).
    k = min(n_model1_worse, n - n_model1_worse)
    p_value = min(1.0, 2 * binom.cdf(k, n, 0.5))

    print(f"p-value: {p_value:.6g}")
    if p_value < alpha:
        print("H0 отвергается: есть статистически значимое различие.")
    else:
        print("Нет оснований отвергнуть H0 о нулевой медиане парных разностей.")

,squared_error_model1,squared_error_model2
0,0.155576,0.156405
1,0.162021,0.160284
2,0.023040,0.023140
3,0.021454,0.021849
4,1.025160,1.025895
...,...,...
20635,0.455104,0.444941
20636,0.047616,0.045013
20637,0.541409,0.532393
20638,0.275312,0.269308


Медиана ошибки model1: 0.171939
Медиана ошибки model2: 0.172138
Ненулевых разностей: 20640
Model1 хуже: 9928; model2 хуже: 10712
p-value: 5.01543e-08
H0 отвергается: есть статистически значимое различие.


In [3]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True, return_X_y=True)
df, target = data[0], data[1] 

In [4]:
from scipy import stats

group_young = df[df['HouseAge'] <= 30]['MedInc']
group_old = df[df['HouseAge'] > 30]['MedInc']

# Выполняем критерий Манна-Уитни-Уилкоксона для сравнения распределений
_, p_value = stats.mannwhitneyu(group_young, group_old, alternative='two-sided')

print(f"p-value: {round(p_value, 2)}")


p-value: 0.0
